# 3.4 Dynamic Prompts (adaptado a lab local)

**Qué se ve aquí:** `@dynamic_prompt` genera el **system prompt en cada llamada** a partir del runtime context (`request.runtime.context`). Aquí: el idioma de respuesta viene en `LanguageContext(user_language=...)`.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

### 📚 Teoría base del módulo 3: el ciclo del agente y dónde se engancha el middleware
`create_agent` es un **bucle**: el modelo piensa → (si pide tools) se ejecutan → el modelo vuelve a pensar → … hasta que responde sin pedir tools.

```
invoke ──► [before_agent] ──► ┌─► [before_model] ─► wrap_model_call( MODELO ) ─► [after_model] ─┐
                              │                                                                  │
                              │   ¿pidió tools? ── no ──► respuesta final ──► [after_agent] ──► fin
                              │        │ sí
                              └── wrap_tool_call( TOOL ) ◄┘
```
| Hook | Corre… | Ejemplo en este módulo |
|---|---|---|
| `before_agent` | 1 vez al empezar el `invoke` | borrar `ToolMessage`s (3.2) |
| `before_model` | antes de **cada** llamada al modelo | `SummarizationMiddleware` (3.2) |
| `wrap_model_call` | **envolviendo** cada llamada al modelo | cambiar modelo (3.4), tools (3.4, 3.5), prompt (`@dynamic_prompt` está hecho sobre esto) |
| `after_model` | después de cada respuesta del modelo | `HumanInTheLoopMiddleware` revisa los `tool_calls` y pausa (3.3) |
| `wrap_tool_call` | envolviendo cada ejecución de tool | reintentos o errores (como el interceptor MCP de M2.4) |

**Analogía:** el agente es un empleado; el middleware son los **supervisores** que revisan su trabajo en momentos fijos (al entrar, antes de hablar con el cliente, antes de enviar un correo…). El empleado no cambia; cambian las reglas que lo rodean.

### 📚 Teoría: prompts dinámicos (`@dynamic_prompt`)
**Idea:** el system prompt no tiene que ser un texto fijo. Se puede **armar en cada llamada** con datos del usuario (idioma, rol, nombre, plan), del state (si ya se autenticó, en 3.5) o de la hora.

- `@dynamic_prompt` es un atajo hecho **sobre `wrap_model_call`**: tu función recibe el `request` y devuelve un `str` (o un `SystemMessage`) que **reemplaza** el system prompt de esa llamada.
- Los datos vienen del **runtime context** (`request.runtime.context`), el mismo de M2.2a: datos **fijos** durante el `invoke`, que el modelo no puede cambiar.

**¿Por qué no poner todo en un solo prompt gigante?** Porque cada instrucción que no aplica es **ruido y tokens**. Con prompts dinámicos, el modelo solo ve lo que corresponde a ese usuario.

| | Runtime context | State |
|---|---|---|
| Quién lo pone | quien llama (`context=...`) | el agente y las tools (`Command(update=...)`) |
| ¿Cambia en el `invoke`? | No | Sí |
| Ejemplo | idioma, rol, credenciales | `authenticated`, mensajes |

### 🧠 Para no confundirse (mnemotecnia)
- **Aquí no hay tools.** Lo que cambia es el **guion** (system prompt) que recibe el modelo **antes de cada llamada**.
- **Analogía del teatro:** el modelo es el **actor**; `@wrap_model_call` es el **asistente de camerino** que, antes de cada escena, puede cambiarle:
  - el **actor** → *dynamic models* (`override(model=...)`)
  - el **guion** → *dynamic prompts* (`@dynamic_prompt`) ← **este notebook**
  - la **utilería** → *dynamic tools* (`override(tools=...)`)
- **Context = Carnet:** lo traes al entrar (`context=LanguageContext("Irish")`), no cambia durante el `invoke` y el modelo no lo puede tocar. **State = Se mueve.**
- **Patrón L-O-H:** **L**eer (`request.runtime.context.user_language`) → **O**verride (aquí lo hace `@dynamic_prompt` por ti, con el texto que devuelves) → **H**andler (llamar al modelo).
- **El guion no se guarda:** el system prompt dinámico se inyecta en cada llamada y **no** queda en `response["messages"]` (por eso `pretty_print` solo muestra Human y Ai). Para verlo: LangSmith (input del run `ChatOllama`) o el "espía" de abajo.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

### 📘 El contexto y la función de prompt
- `LanguageContext` es un `dataclass`: define **qué datos** de contexto acepta el agente (aquí, `user_language`, por defecto `English`).
- `user_language_prompt` lee `request.runtime.context.user_language` y arma el prompt. Si no es inglés, agrega *"only respond in …"*.

In [2]:
# Sin cambios
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

### 📘 `context_schema` + `middleware`
`context_schema=LanguageContext` le dice al agente qué forma tiene el contexto; `middleware=[user_language_prompt]` activa el prompt dinámico.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     context_schema=LanguageContext,
#     middleware=[user_language_prompt]
# )

In [3]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

### 📘 Mismo mensaje, distinto contexto
La pregunta es idéntica (*"Hello, how are you?"*); solo cambia `context=LanguageContext(...)`. El modelo responde en irlandés, español y francés.
> Con un modelo local chico, el irlandés puede salir con errores: el prompt funciona, pero el modelo sabe menos de ese idioma.

In [4]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia is Muire! Tá mé go maith, go raibh maith agat. Agus tú?


In [5]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hello, how are you?
================================== Ai Message ==================================

Dia is Muire! Tá mé go maith, go raibh maith agat. Agus tú?


In [6]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

Hola, estoy bien, gracias. ¿Y tú?


In [7]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hello, how are you?
================================== Ai Message ==================================

Hola, estoy bien, gracias. ¿Y tú?


In [8]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Je vais bien, merci. Et vous, comment allez-vous ?


In [9]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hello, how are you?
================================== Ai Message ==================================

Bonjour ! Je vais bien, merci. Et vous, comment allez-vous ?


## 🕵️ EXTRA: ver el system prompt que **realmente** recibe el modelo
Un segundo middleware "espía" (`@wrap_model_call`) que solo imprime `request.system_message` y sigue.
**Orden importa:** en `middleware=[user_language_prompt, espia]` el primero **envuelve** al segundo, así que el espía ve el prompt **ya cambiado**.
Fíjate al final: en `messages` solo hay `human` y `ai`; el system prompt **no** se guarda en el state.

In [ ]:
# 🟢 EXTRA (aprendizaje): espía del system prompt
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def espia(request, handler):
    print("🕵️ system prompt que recibe el modelo →", request.system_message.content if request.system_message else None)
    return handler(request)

agent_espia = create_agent(
    model=MODEL,
    context_schema=LanguageContext,
    middleware=[user_language_prompt, espia],   # primero arma el prompt, después el espía lo lee
)

for idioma in ["English", "Spanish", "Irish"]:
    r = agent_espia.invoke(
        {"messages": [HumanMessage(content="Hello, how are you?")]},
        context=LanguageContext(user_language=idioma),
    )
    print(f"   {idioma:<8} → {r['messages'][-1].content}")
    print("   tipos guardados en messages:", [m.type for m in r["messages"]], "\n")